# Practical Guide to Task Automation using ChatGPT & Python

[Article](https://todatabeyond.com/blog/practical-guide-to-task-automation-using-chatgpt-and-python). Original examples are preserved in the nonexecuting appendix below.

Uses free public services. Only submit audio/text you own and may send to Google. Weather and reference currency rates change. Email tests bind only to loopback and do not relay mail. Install `pip install SpeechRecognition==3.17.0 gTTS==2.5.4 aiosmtpd==1.4.6 requests==2.33.0 certifi==2026.10.23` (or use apps/python-task-automation/requirements.txt in a clone); FFmpeg must also be installed (Colab normally includes it).

In [ ]:
import sys, os, json, runpy, tempfile, subprocess, shutil, urllib.request
from pathlib import Path
from importlib.metadata import version
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
work = Path(tempfile.mkdtemp(prefix="tdb-automation-"))
app = Path.cwd() / "apps" / "python-task-automation"
if not app.is_dir():
    app = work / "scripts"
    app.mkdir()
    base = "https://raw.githubusercontent.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/main/apps/python-task-automation/"
    for filename in ["audio_text.py", "text_audio.py", "weather_report.py", "currency_converter.py", "automate_email.py"]:
        urllib.request.urlretrieve(base + filename, app / filename)
def run_script(name, *args):
    result = subprocess.run([sys.executable, str(app / name), *map(str, args)],
                            capture_output=True, text=True, timeout=90, check=True)
    print(result.stdout.strip())
    return result
print("Python:", sys.version.split()[0])
for package in ["SpeechRecognition", "gTTS", "aiosmtpd", "requests", "certifi"]:
    print(package, version(package))
print("Five scripts ready; all generated files stay in a temporary workspace.")

In [ ]:
phrase = "Hello world. This is a Python automation example."
text_path = work / "example.txt"
text_path.write_text(phrase, encoding="utf-8")
tts_result = run_script("text_audio.py", text_path, work / "example.mp3")
assert (work / "example.mp3").stat().st_size > 0

In [ ]:
if not shutil.which("ffmpeg"):
    raise RuntimeError("Install FFmpeg before converting the synthetic MP3 to WAV")
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(work / "example.mp3"),
                "-ac", "1", "-ar", "16000", str(work / "example.wav")], check=True)
transcript = run_script("audio_text.py", work / "example.wav").stdout.strip()
assert transcript.lower().replace(".", "") == phrase.lower().replace(".", "")
print("PASS: synthetic phrase transcribed correctly over HTTPS.")

In [ ]:
weather_result = run_script("weather_report.py", "Cairo")
assert "Cairo" in weather_result.stdout
assert "\x1b[" not in weather_result.stdout

In [ ]:
currency_result = run_script("currency_converter.py", "100", "USD", "EUR")
assert "100 USD =" in currency_result.stdout and "Rate date:" in currency_result.stdout

In [ ]:
import socket
from email import policy
from email.parser import BytesParser
from aiosmtpd.controller import Controller
class Capture:
    def __init__(self): self.messages = []
    async def handle_DATA(self, server, session, envelope):
        self.messages.append((envelope.mail_from, list(envelope.rcpt_tos), envelope.content))
        return "250 Captured locally; not relayed"
capture = Capture()
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    port = sock.getsockname()[1]
controller = Controller(capture, hostname="127.0.0.1", port=port)
controller.start()
try:
    email_result = run_script("automate_email.py", "--port", port)
finally:
    controller.stop()
assert len(capture.messages) == 1
sender, recipients, raw = capture.messages[0]
mail = BytesParser(policy=policy.default).parsebytes(raw)
assert sender == "sender@example.invalid"
assert recipients == ["recipient@example.invalid"]
assert mail["Subject"] == "Automation example"
assert mail.get_content().strip() == "Hello from the local automation tutorial."
print("PASS: envelope, headers and message body captured on loopback only; no real email sent.")

In [ ]:
from decimal import Decimal
currency = runpy.run_path(str(app / "currency_converter.py"))["convert_currency"]
assert currency("1.23", "usd", "USD") == (Decimal("1.23"), "same currency", Decimal("1"))
for amount, base, quote in [("NaN", "USD", "EUR"), ("-1", "USD", "EUR"), ("1", "US", "EUR")]:
    try: currency(amount, base, quote)
    except ValueError: pass
    else: raise AssertionError("Invalid currency input accepted")
class FakeResponse:
    def raise_for_status(self): pass
    def json(self, **kwargs):
        return {"base": "USD", "quote": "EUR", "rate": Decimal("0.89326"), "date": "2026-10-27"}
class FakeSession:
    def get(self, *args, **kwargs): return FakeResponse()
assert currency("100", "USD", "EUR", FakeSession())[0] == Decimal("89.32600")
weather = runpy.run_path(str(app / "weather_report.py"))["weather_report"]
try: weather(" ")
except ValueError: pass
else: raise AssertionError("Empty weather location accepted")
send = runpy.run_path(str(app / "automate_email.py"))["send_email"]
for kwargs in [{"smtp_host": "smtp.gmail.com"}, {"subject": "bad\nheader"}, {"recipient_emails": []}]:
    args = dict(sender_email="sender@example.invalid", recipient_emails=["recipient@example.invalid"], subject="Test", message="Body")
    args.update(kwargs)
    try: send(**args)
    except ValueError: pass
    else: raise AssertionError("Unsafe email accepted")
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    closed_port = sock.getsockname()[1]
try: send("sender@example.invalid", ["recipient@example.invalid"], "Test", "Body", smtp_port=closed_port)
except ConnectionRefusedError: pass
else: raise AssertionError("Expected connection failure")
empty = work / "empty.txt"
empty.write_text("", encoding="utf-8")
tts = runpy.run_path(str(app / "text_audio.py"))["text_to_audio"]
try: tts(empty, work / "empty.mp3")
except ValueError: pass
else: raise AssertionError("Empty TTS input accepted")
for name in ["audio_text.py", "text_audio.py", "weather_report.py", "currency_converter.py"]:
    p = subprocess.run([sys.executable, str(app / name)], capture_output=True, text=True)
    assert p.returncode == 2 and "usage:" in p.stderr
print("PASS: Decimal arithmetic, same-currency behavior, invalid inputs, local SMTP policy, header validation, failed connection cleanup, empty TTS input and all four CLI usage errors.")

## Complete maintained CLI scripts

### audio_text.py

```python
import argparse
import os
from pathlib import Path
import certifi
import speech_recognition as sr

def transcribe_audio(audio_file_path):
    # Use a trusted CA bundle, never disable certificate verification.
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    recognizer = sr.Recognizer()
    recognizer.operation_timeout = 30
    with sr.AudioFile(str(audio_file_path)) as source:
        audio = recognizer.record(source)
    return recognizer.recognize_google(
        audio, endpoint="https://www.google.com/speech-api/v2/recognize")

def main():
    parser = argparse.ArgumentParser(description="Transcribe an owned WAV/AIFF/FLAC audio file using Google speech recognition.")
    parser.add_argument("audio_file", type=Path)
    args = parser.parse_args()
    try:
        print(transcribe_audio(args.audio_file))
    except (OSError, ValueError, sr.UnknownValueError, sr.RequestError) as exc:
        parser.exit(1, f"Transcription failed: {exc}\n")

if __name__ == "__main__":
    main()
```

### text_audio.py

```python
import argparse
from pathlib import Path
from gtts import gTTS, gTTSError

def text_to_audio(text_file_path, output_audio_path):
    text = Path(text_file_path).read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError("Input text is empty")
    gTTS(text=text, lang="en", timeout=(10, 30)).save(str(output_audio_path))
    return Path(output_audio_path)

def main():
    parser = argparse.ArgumentParser(description="Convert owned English text to MP3 using Google Translate TTS.")
    parser.add_argument("text_file", type=Path)
    parser.add_argument("output_audio", type=Path)
    args = parser.parse_args()
    try:
        output = text_to_audio(args.text_file, args.output_audio)
        print(f"Created {output.name}: {output.stat().st_size} bytes")
    except (OSError, ValueError, gTTSError) as exc:
        parser.exit(1, f"Text-to-speech failed: {exc}\n")

if __name__ == "__main__":
    main()
```

### weather_report.py

```python
import argparse
from urllib.parse import quote
import requests

def weather_report(location, session=None):
    location = location.strip()
    if not location or any(ord(c) < 32 for c in location):
        raise ValueError("Provide a nonempty location without control characters")
    http = session or requests
    response = http.get("https://wttr.in/" + quote(location, safe=""),
                        params={"T": ""}, timeout=(10, 30),
                        headers={"User-Agent": "curl/8.0"})
    response.raise_for_status()
    return response.text

def main():
    parser = argparse.ArgumentParser(description="Print a plain-text weather report from wttr.in.")
    parser.add_argument("location")
    args = parser.parse_args()
    try:
        print(weather_report(args.location))
    except (ValueError, requests.RequestException) as exc:
        parser.exit(1, f"Weather request failed: {exc}\n")

if __name__ == "__main__":
    main()
```

### currency_converter.py

```python
import argparse
import re
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import requests

def convert_currency(amount, from_currency, to_currency, session=None):
    amount = Decimal(str(amount))
    if not amount.is_finite() or amount < 0:
        raise ValueError("Amount must be finite and nonnegative")
    base, quote = from_currency.upper(), to_currency.upper()
    if not all(re.fullmatch(r"[A-Z]{3}", c) for c in (base, quote)):
        raise ValueError("Use three-letter ISO currency codes")
    if base == quote:
        return amount, "same currency", Decimal("1")
    response = (session or requests).get(
        f"https://api.frankfurter.dev/v2/rate/{base}/{quote}", timeout=(10, 30))
    response.raise_for_status()
    data = response.json(parse_float=Decimal)
    if data["base"] != base or data["quote"] != quote:
        raise ValueError("Provider returned the wrong currency pair")
    rate = Decimal(str(data["rate"]))
    if not rate.is_finite() or rate <= 0:
        raise ValueError("Provider returned an invalid rate")
    return amount * rate, data["date"], rate

def main():
    parser = argparse.ArgumentParser(description="Convert using dated Frankfurter reference rates, not transaction quotes.")
    parser.add_argument("amount")
    parser.add_argument("from_currency")
    parser.add_argument("to_currency")
    args = parser.parse_args()
    try:
        result, date, rate = convert_currency(args.amount, args.from_currency, args.to_currency)
        rounded = result.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        print(f"{args.amount} {args.from_currency.upper()} = {rounded} {args.to_currency.upper()}")
        print(f"Rate date: {date}; rate: {rate}")
    except (ValueError, InvalidOperation, KeyError, requests.RequestException) as exc:
        parser.exit(1, f"Currency conversion failed: {exc}\n")

if __name__ == "__main__":
    main()
```

### automate_email.py

```python
import argparse
import os
import smtplib
import ssl
from email.message import EmailMessage
from email.utils import parseaddr

def send_email(sender_email, recipient_emails, subject, message, *,
               smtp_host="127.0.0.1", smtp_port=8025, username=None,
               password=None, use_starttls=False, allow_remote=False):
    recipients = list(recipient_emails)
    if not recipients:
        raise ValueError("Provide at least one recipient")
    for address in [sender_email, *recipients]:
        if not isinstance(address, str) or any(c in address for c in "\r\n"):
            raise ValueError("Invalid address")
        if parseaddr(address)[1] != address or "@" not in address:
            raise ValueError("Provide bare email addresses")
    if any(c in subject for c in "\r\n"):
        raise ValueError("Invalid subject")
    local = smtp_host in {"127.0.0.1", "::1"}
    if not local and not (allow_remote and use_starttls and username and password):
        raise ValueError("Remote email requires explicit opt-in, TLS and credentials")
    if bool(username) != bool(password):
        raise ValueError("Provide both username and password")
    mail = EmailMessage()
    mail["From"], mail["To"], mail["Subject"] = sender_email, ", ".join(recipients), subject
    mail.set_content(message)
    with smtplib.SMTP(smtp_host, smtp_port, timeout=15) as server:
        server.ehlo()
        if use_starttls:
            server.starttls(context=ssl.create_default_context())
            server.ehlo()
        if username:
            server.login(username, password)
        refused = server.send_message(mail, from_addr=sender_email, to_addrs=recipients)
        if refused:
            raise smtplib.SMTPRecipientsRefused(refused)
    return len(recipients)

def main():
    parser = argparse.ArgumentParser(description="Send to a local SMTP capture by default; remote delivery requires explicit opt-in.")
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=8025)
    parser.add_argument("--allow-remote", action="store_true")
    parser.add_argument("--starttls", action="store_true")
    parser.add_argument("--sender", default="sender@example.invalid")
    parser.add_argument("--to", nargs="+", default=["recipient@example.invalid"])
    parser.add_argument("--subject", default="Automation example")
    parser.add_argument("--message", default="Hello from the local automation tutorial.")
    args = parser.parse_args()
    try:
        count = send_email(args.sender, args.to, args.subject, args.message,
            smtp_host=args.host, smtp_port=args.port,
            username=os.getenv("SMTP_USERNAME"), password=os.getenv("SMTP_PASSWORD"),
            use_starttls=args.starttls, allow_remote=args.allow_remote)
        print(f"Accepted by SMTP capture/server: {count} recipient(s)")
    except (OSError, ValueError, smtplib.SMTPException) as exc:
        parser.exit(1, f"Email delivery failed: {exc}\n")

if __name__ == "__main__":
    main()
```

## Original archive appendix — not executed

The original code, prompts and transcripts below are retained verbatim for comparison. Historical SDK calls, placeholders and setup instructions are not the runnable path.

### Original block 1

```text
import sys
import speech_recognition as sr

def convert_audio_to_text(audio_file):
    # Initialize the recognizer
    r = sr.Recognizer()
    
    # Load the audio file
    with sr.AudioFile(audio_file) as source:
        # Read the audio data from the file
        audio_data = r.record(source)
        
        try:
            # Use the recognizer to convert audio to text
            text = r.recognize_google(audio_data)
            return text
        except sr.UnknownValueError:
            print("Speech recognition could not understand audio")
        except sr.RequestError as e:
            print(f"Could not request results from the speech recognition service; {e}")
    
    return None

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python script.py <audio_file_path>")
        sys.exit(1)
    
    audio_file_path = sys.argv[1]
    
    # Convert audio to text
    result = convert_audio_to_text(audio_file_path)
    
    if result:
        # Print the converted text
        print(result)
```

### Original block 2

```text
python script.py path/to/audio/file.wav
```

### Original block 3

```text
pip install SpeechRecognition
pip install pydub
pip install ffmpeg-python
```

### Original block 4

```text
import sys
from gtts import gTTS

def convert_text_to_audio(text, output_file):
    # Create a gTTS object with the text and desired language
    tts = gTTS(text=text, lang='en')

    # Save the audio to a file
    tts.save(output_file)
    print(f"Audio file saved as {output_file}")

if __name__ == "__main__":
    if len(sys.argv) < 3:
        print("Usage: python script.py <text_file_path> <output_audio_file_path>")
        sys.exit(1)

    text_file_path = sys.argv[1]
    output_audio_file = sys.argv[2]

    # Read the text from the file
    with open(text_file_path, 'r') as file:
        text = file.read()

    # Convert text to audio
    convert_text_to_audio(text, output_audio_file)
```

### Original block 5

```text
python script.py path/to/text/file.txt output/audio/file.mp3
```

### Original block 6

```text
pip install gtts
```

### Original block 7

```text
Can you write me a python script to build weather report that you run in the command line and mention the needed pacakges to download
```

### Original block 8

```text
pip install requests
```

### Original block 9

```text
import sys
import requests
resp = requests.get(f'https://wttr.in/{sys.argv[1].replace(" ", "+")}')
print(resp.text)
```

### Original block 10

```text
python weather_report.py <location>
```

### Original block 11

```text
python weather_report.py "Cairo"
```

### Original block 12

```text
Develope a converting currencies script using python that run through the command line and mention the needed pacakges.
```

### Original block 13

```text
pip install forex-python
```

### Original block 14

```text
import argparse
from forex_python.converter import CurrencyRates, CurrencyCodes

# Initialize the currency rates and codes objects
c = CurrencyRates()
codes = CurrencyCodes()

# Set up the argument parser
parser = argparse.ArgumentParser(description="Convert currencies using forex-python")
parser.add_argument("amount", type=float, help="The amount to convert")
parser.add_argument("from_currency", type=str, help="The currency you're converting from (e.g. USD)", metavar="from")
parser.add_argument("to_currency", type=str, help="The currency you're converting to (e.g. EUR)", metavar="to")

# Parse the arguments
args = parser.parse_args()

# Get the amount and currencies from the arguments
amount = args.amount
from_currency = args.from_currency.upper()
to_currency = args.to_currency.upper()

# Get the currency symbols for display purposes
from_symbol = codes.get_symbol(from_currency)
to_symbol = codes.get_symbol(to_currency)

# Convert the currencies
result = c.convert(from_currency, to_currency, amount)

# Print the result
print(f"{from_symbol}{amount:.2f} {from_currency} is equal to {to_symbol}{result:.2f} {to_currency}")
```

### Original block 15

```text
python currency_converter.py 100 USD EUR
```

### Original block 16

```text
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

def send_email(sender_email, sender_password, recipient_emails, subject, message):
    # Set up the SMTP server details
    smtp_server = 'smtp.gmail.com'
    smtp_port = 587

    # Create a multipart message
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = ', '.join(recipient_emails)
    msg['Subject'] = subject

    # Attach the message as plain text
    msg.attach(MIMEText(message, 'plain'))

    try:
        # Connect to the SMTP server
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()

        # Log in to the sender's email account
        server.login(sender_email, sender_password)

        # Send the email
        server.send_message(msg)
        print('Email sent successfully')

    except smtplib.SMTPException as e:
        print(f'Error: Unable to send email. {str(e)}')

    finally:
        # Disconnect from the SMTP server
        server.quit()

# Provide the email details
sender_email = 'your_email@gmail.com'
sender_password = 'your_password'
recipient_emails = ['recipient1@example.com', 'recipient2@example.com']
subject = 'Automated Email'
message = 'This is an automated email sent using Python.'

# Send the email
send_email(sender_email, sender_password, recipient_emails, subject, message)
```

### Original prompt/transcript 1

```text
Write a python script to Automate Converting Audio Files into Text Files and make it run through the command line
```

### Original prompt/transcript 2

```text
write a Python script to Automate Converting Text Files into audio files make it run through the command line
```

### Original prompt/transcript 3

```text
can you write a python script to send automated mails
```